# Capítulo 9: Classificação

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 4 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> A visão geral deste capítulo ainda será escrita.

## Seções

| Seção | Tópico |
|---|---|
| [9.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/01-por-que-nao-regressao-linear.html) | Por que Não Regressão Linear |
| [9.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-regressao-logistica.html) | Regressão Logística |
| [9.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/03-logistica-multinomial.html) | Logística Multinomial |
| [9.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/04-modelos-generativos.html) | Modelos Generativos: LDA, QDA e Naive Bayes |
| [9.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/05-avaliando-um-classificador.html) | Avaliando um Classificador |
| [9.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/06-comparando-os-metodos.html) | Comparando os Métodos |

## Por que Não Regressão Linear

> **📌 Nota**
>
> Esta seção corresponde às seções 4.1 e 4.2 de James et al. (2023).

O capítulo anterior tratou sempre do mesmo tipo de pergunta: dado um conjunto de preditores, que número a resposta deve assumir — vendas em milhares de unidades, consumo em milhas por galão. `Default` muda o tipo da resposta. Dez mil clientes de cartão de crédito, e a pergunta agora é se um cliente fica inadimplente ou não: `inadimplente` não é uma quantidade, é uma categoria, `sim` ou `não`. `saldo`, `renda` e `estudante` (`sim`/`não`) continuam preditores de sempre; o que muda é o alvo, e é essa mudança que o resto do capítulo resolve.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression

plt.style.use("estilo-figuras.mplstyle")

### O `Default`: dez mil clientes, um alvo qualitativo

In [ ]:
base = pd.read_csv("dados/Default.csv")
base.shape, base.columns.tolist()

Dez mil linhas, quatro colunas: `inadimplente` é o alvo; `estudante` é uma segunda categórica; `saldo` (quanto o cliente deve no cartão) e `renda` (a renda anual do cliente) são numéricas, as duas em dólares.

In [ ]:
n_sim = int((base["inadimplente"] == "sim").sum())
n_nao = int((base["inadimplente"] == "não").sum())
proporcao_sim = n_sim / len(base)

n_sim, n_nao, round(proporcao_sim, 4)

333 dos 10.000 clientes ficam inadimplentes; 9.667, não. A proporção, 0,0333, é baixa: menos de um em trinta. `Default` é um conjunto desequilibrado, e um classificador que sempre responder "não" já acerta a maioria — um ponto que volta na seção 9.5, quando "acerta a maioria" deixa de bastar como medida.

### O saldo separa; a renda, não

In [ ]:
# Figura: `Default`: saldo e renda de 10.000 clientes, com quem ficou inadimplente (`sim`) destacado sobre quem não ficou (`não`). Ao lado, os mesmos dois grupos em caixas — quartil 25%, mediana e quartil 75% — para saldo e para renda.
nao = base[base["inadimplente"] == "não"]
sim = base[base["inadimplente"] == "sim"]

def desenha_caixas(ax, dados_nao, dados_sim, rotulo_y):
    for dados, posicao, cor in [(dados_nao, 1, "C0"), (dados_sim, 2, "C1")]:
        propriedades = dict(color=cor, linewidth=1.6)
        ax.boxplot(
            dados, positions=[posicao], widths=0.5,
            boxprops=propriedades, whiskerprops=propriedades,
            capprops=propriedades, medianprops=propriedades,
            flierprops=dict(markeredgecolor=cor, markersize=3, alpha=0.5),
        )
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["não", "sim"])
    ax.set_xlabel("inadimplente")
    ax.set_ylabel(rotulo_y)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4.4))

ax1.scatter(nao["saldo"], nao["renda"], color="C0", s=10, alpha=0.35, label="não")
ax1.scatter(sim["saldo"], sim["renda"], color="C1", s=14, alpha=0.85, label="sim", zorder=3)
ax1.set_xlabel("saldo (dólares)")
ax1.set_ylabel("renda (dólares)")
ax1.legend(title="inadimplente", loc="upper right", markerscale=1.6)

desenha_caixas(ax2, nao["saldo"], sim["saldo"], "saldo (dólares)")
desenha_caixas(ax3, nao["renda"], sim["renda"], "renda (dólares)")

plt.tight_layout()
plt.show()

A dispersão à esquerda já sugere uma leitura: os pontos laranja (`sim`) se acumulam à direita, em saldos altos; os azuis (`não`), à esquerda. Em renda, as duas cores se misturam ao longo de todo o eixo vertical. As caixas ao lado conferem essa leitura em vez de só ilustrá-la:

In [ ]:
q1_saldo_nao = float(nao["saldo"].quantile(0.25))
q3_saldo_nao = float(nao["saldo"].quantile(0.75))
q1_saldo_sim = float(sim["saldo"].quantile(0.25))
q3_saldo_sim = float(sim["saldo"].quantile(0.75))
sobrepoe_saldo = not (q3_saldo_nao < q1_saldo_sim or q3_saldo_sim < q1_saldo_nao)

q1_renda_nao = float(nao["renda"].quantile(0.25))
q3_renda_nao = float(nao["renda"].quantile(0.75))
q1_renda_sim = float(sim["renda"].quantile(0.25))
q3_renda_sim = float(sim["renda"].quantile(0.75))
sobrepoe_renda = not (q3_renda_nao < q1_renda_sim or q3_renda_sim < q1_renda_nao)

(
    round(q1_saldo_nao, 2), round(q3_saldo_nao, 2), round(q1_saldo_sim, 2), round(q3_saldo_sim, 2), sobrepoe_saldo,
    round(q1_renda_nao, 2), round(q3_renda_nao, 2), round(q1_renda_sim, 2), round(q3_renda_sim, 2), sobrepoe_renda,
)

Em `saldo`, o quartil 75% de quem não ficou inadimplente (1.128,25) fica abaixo do quartil 25% de quem ficou (1.511,61): as duas caixas nem se tocam, `sobrepoe_saldo` sai `False`. Em `renda`, o intervalo de quem não ficou inadimplente vai de 21.405,06 a 43.823,76, e o de quem ficou, de 19.027,51 a 43.067,33 — um contido quase inteiramente dentro do outro, `sobrepoe_renda` sai `True`. O saldo separa os dois grupos; a renda, não.

### Por que a reta não serve para probabilidade

Recodificando `inadimplente` como 0 (`não`) e 1 (`sim`), nada impede de ajustar a mesma `LinearRegression` do capítulo anterior sobre esse alvo — o método não sabe, e não pergunta, se `y` é uma venda em milhares de unidades ou uma categoria disfarçada de número. O que ele devolve, porém, deixa de ser uma venda prevista e passa a ser lido como uma probabilidade prevista: a de que aquele cliente fique inadimplente.

In [ ]:
y = (base["inadimplente"] == "sim").astype(int)
X = base[["saldo"]]

reta = LinearRegression().fit(X, y)
previsoes = reta.predict(X)

n_negativas = int((previsoes < 0).sum())
minimo_previsto = float(previsoes.min())

n_negativas, round(minimo_previsto, 4)

3.123 dos 10.000 clientes recebem da reta uma previsão negativa — quase um em cada três. O mínimo, -0,0752, é a previsão para quem tem o menor saldo do conjunto. Probabilidade negativa não tem leitura possível: nenhum cliente tem chance "menos que zero por cento" de ficar inadimplente, e é esse o problema — não que a reta erre feio de vez em quando, mas que uma fração inteira das suas previsões caia fora do intervalo em que uma probabilidade pode existir.

### A curva que não sai de [0, 1]

In [ ]:
# Figura: Saldo contra a previsão de inadimplência, para os mesmos clientes de `Default`. Esquerda: a reta ajustada acima, que cruza a faixa sombreada — o intervalo [0, 1] — e sai por baixo dela. Direita: uma curva logística ajustada aos mesmos dados; os traços no alto e embaixo marcam, para cada cliente, se ele ficou inadimplente (sim, no topo) ou não (não, embaixo).
logistica = LogisticRegression().fit(X, y)

grade_saldo = pd.DataFrame({"saldo": np.linspace(base["saldo"].min(), base["saldo"].max(), 300)})
reta_grade = reta.predict(grade_saldo)
prob_logistica_grade = logistica.predict_proba(grade_saldo)[:, 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)

for ax in (ax1, ax2):
    ax.axhspan(0, 1, color="C2", alpha=0.10)
    ax.plot(nao["saldo"], np.zeros(len(nao)), "|", color="C0", alpha=0.4, markersize=8)
    ax.plot(sim["saldo"], np.ones(len(sim)), "|", color="C1", alpha=0.6, markersize=8)
    ax.set_xlabel("saldo (dólares)")

ax1.plot(grade_saldo["saldo"], reta_grade, color="C3", linewidth=2.2)
ax1.scatter([0.0], [minimo_previsto], color="C3", s=40, zorder=3)
ax1.annotate(
    f"mínimo: {minimo_previsto:.3f}",
    xy=(0.0, minimo_previsto),
    xytext=(14, -6),
    textcoords="offset points",
    fontsize=8,
)
ax1.set_ylabel("previsão / probabilidade de inadimplência")
ax1.set_title("reta")

ax2.plot(grade_saldo["saldo"], prob_logistica_grade, color="C3", linewidth=2.2)
ax2.set_title("curva logística")

plt.tight_layout()
plt.show()

A reta (esquerda) atravessa a faixa sombreada e sai por baixo dela para saldos próximos de zero — a mesma previsão negativa medida acima, agora como curva. A curva logística (direita), ajustada aos mesmos clientes, tem outro formato: um S que se aproxima de 0 e de 1 sem nunca alcançá-los.

In [ ]:
minimo_logistica = float(prob_logistica_grade.min())
maximo_logistica = float(prob_logistica_grade.max())
n_fora_logistica = int(((prob_logistica_grade < 0) | (prob_logistica_grade > 1)).sum())

round(minimo_logistica, 5), round(maximo_logistica, 4), n_fora_logistica

Sobre a mesma grade de 300 valores de saldo usada na figura, a curva logística vai de 0,00002 a 0,9810 — perto das bordas, mas sem cruzá-las — e `n_fora_logistica` conta zero pontos fora de [0, 1]. Como a curva se ajusta e por que ela tem esse formato é assunto da próxima seção; o que importa aqui é só que ela existe, e que resolve o problema que a reta tem.

### Mais de duas classes, e a ordem que a codificação inventa

O problema muda de figura, mas não desaparece, quando o alvo tem mais de duas categorias. Um paciente chega ao pronto-socorro com sintomas que apontam para um de três diagnósticos — derrame, overdose ou convulsão —, e codificar isso como `Y = 1` para derrame, `2` para overdose e `3` para convulsão impõe duas coisas que a lista de diagnósticos não tinha: uma ordem entre os três, e a afirmação de que a distância entre derrame e overdose é a mesma que entre overdose e convulsão. Trocar a ordem — `1` para convulsão, `2` para derrame, `3` para overdose — é uma codificação igualmente válida, e produz um modelo linear diferente do primeiro; nenhuma das duas é mais correta, porque não existe uma escala numérica por trás do diagnóstico que a codificação possa recuperar. Reta ou curva, uma resposta com mais de duas categorias sem ordem natural pede outro tratamento — o que a seção 9.3 faz.

## Regressão Logística

> **📌 Nota**
>
> Esta seção corresponde às seções 4.3.1, 4.3.2, 4.3.3 e 4.3.4 de James et al. (2023).

A seção anterior deixou a curva em S pronta, sem dizer de onde ela vem nem como um coeficiente se lê nela. A regressão logística resolve as duas coisas: ajusta essa curva a dado real, e dá um coeficiente por preditor — só que lido de um jeito diferente do que o capítulo 8 ensinou para a reta.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

plt.style.use("estilo-figuras.mplstyle")

base = pd.read_csv("dados/Default.csv")
y = (base["inadimplente"] == "sim").astype(int)

### A função logística

Para um preditor $X$ e coeficientes $\beta_0$ e $\beta_1$, a regressão logística modela a probabilidade como

$$
p(X) = \frac{e^{\beta_0 + \beta_1 X}}{1 + e^{\beta_0 + \beta_1 X}}.
$$

O denominador é sempre o numerador mais 1, e os dois são positivos para qualquer $\beta_0$, $\beta_1$ e $X$ reais — a fração fica presa em (0, 1) pela forma da conta, não pelo ajuste. É essa propriedade, e não o valor de nenhum coeficiente, que resolve o problema da seção anterior.

In [ ]:
grade_z = np.linspace(-10, 10, 400)
p_z = np.exp(grade_z) / (1 + np.exp(grade_z))

p_extremo_baixo = float(p_z.min())
p_extremo_alto = float(p_z.max())

f"{p_extremo_baixo:.6f}", f"{p_extremo_alto:.6f}"

Em $\beta_0 + \beta_1 X = -10$, a curva vale 0,000045; em $+10$, vale 0,999955 — perto das bordas de [0, 1], mas sem nunca as tocar, por mais que $\beta_0 + \beta_1 X$ se afaste de zero.

In [ ]:
# Figura: A função logística, para z = β0 + β1X entre -10 e 10. A curva se aproxima de 0 e de 1 sem nunca alcançá-los, e é essa forma que sustenta todo ajuste desta seção.
fig, ax = plt.subplots()
ax.axhspan(0, 1, color="C2", alpha=0.10)
ax.plot(grade_z, p_z, color="C0", linewidth=2.2)
ax.set_xlabel("β0 + β1·X")
ax.set_ylabel("p(X)")
plt.tight_layout()
plt.show()

### O ajuste sobre saldo

In [ ]:
modelo_saldo = LogisticRegression(C=np.inf, max_iter=10000).fit(base[["saldo"]], y)

intercepto_saldo = float(modelo_saldo.intercept_[0])
coef_saldo = float(modelo_saldo.coef_[0, 0])

round(intercepto_saldo, 4), round(coef_saldo, 4)

`C=np.inf` desliga a penalização que o `LogisticRegression` aplica por padrão, deixando o ajuste livre para maximizar a verossimilhança sem encolher nenhum coeficiente. Sobre `saldo` sozinho, o ajuste dá intercepto -10,6513 e coeficiente 0,0055.

### A leitura do coeficiente: log-chance, não probabilidade

$$
\log\left(\frac{p(X)}{1-p(X)}\right) = \beta_0 + \beta_1 X.
$$

> **🔷 Conceito**
>
> Um aumento de uma unidade em $X$ soma $\beta_1$ à **log-chance** de $Y = 1$, não à probabilidade $p(X)$. Como a relação entre $p(X)$ e $X$ não é uma reta, o efeito de $X$ sobre a probabilidade depende de onde $X$ já está — diferente da regressão linear do capítulo 8, em que $\beta_1$ valia o mesmo incremento em qualquer ponto.

Sobre `saldo`, cada dólar a mais soma sempre os mesmos 0,0055 à log-chance de inadimplência — mas o efeito sobre a probabilidade não é constante:

In [ ]:
grade_previsao = pd.DataFrame({"saldo": [1000, 2000]})
prob_1000, prob_2000 = modelo_saldo.predict_proba(grade_previsao)[:, 1]

round(float(prob_1000) * 100, 2), round(float(prob_2000) * 100, 2)

Para um saldo de mil dólares, a probabilidade prevista de inadimplência é 0,58%; para um saldo de dois mil dólares — o dobro do primeiro —, ela não dobra: sobe para 58,58%. É essa não linearidade que a curva em S carrega, e que a reta da seção anterior não tinha como representar.

### O paradoxo do estudante

In [ ]:
estudantes = base[base["estudante"] == "sim"]
nao_estudantes = base[base["estudante"] == "não"]

taxa_estudante = float((estudantes["inadimplente"] == "sim").mean())
taxa_nao_estudante = float((nao_estudantes["inadimplente"] == "sim").mean())

round(taxa_estudante * 100, 2), round(taxa_nao_estudante * 100, 2)

Entre estudantes, 4,31% ficam inadimplentes; entre os demais, 2,92% — olhado sozinho, o estudante é o cliente mais arriscado dos dois.

In [ ]:
X_estudante = (base["estudante"] == "sim").astype(int).to_frame("estudante_sim")
modelo_estudante = LogisticRegression(C=np.inf, max_iter=10000).fit(X_estudante, y)

coef_estudante_sozinho = float(modelo_estudante.coef_[0, 0])
round(coef_estudante_sozinho, 4)

Um modelo logístico que usa só `estudante` como preditor confirma essa leitura: o coeficiente sai positivo, 0,4014 — ser estudante soma à log-chance de inadimplência.

In [ ]:
X_multiplo = base[["saldo", "renda"]].copy()
X_multiplo["estudante_sim"] = (base["estudante"] == "sim").astype(int)

modelo_multiplo = LogisticRegression(C=np.inf, max_iter=10000).fit(X_multiplo, y)
coeficientes_multiplo = pd.Series(modelo_multiplo.coef_[0], index=modelo_multiplo.feature_names_in_)
intercepto_multiplo = float(modelo_multiplo.intercept_[0])

(
    round(intercepto_multiplo, 4),
    round(float(coeficientes_multiplo["saldo"]), 6),
    f"{coeficientes_multiplo['renda']:.3e}",
    round(float(coeficientes_multiplo["estudante_sim"]), 4),
)

Juntar `saldo`, `renda` e `estudante` na mesma equação reverte o sinal: o coeficiente de estudante cai para -0,6468. `renda` quase não muda a conta (3,033e-06), mas `saldo` (0,005737) e `estudante` pesam — e o de `estudante` trocou de sinal por completo. É o mesmo tipo de reviravolta que a seção 8.3 viu no coeficiente de jornal: positivo sozinho, outra coisa depois que os preditores certos entram na mesma equação.

In [ ]:
saldo_medio_estudante = float(estudantes["saldo"].mean())
saldo_medio_nao_estudante = float(nao_estudantes["saldo"].mean())

round(saldo_medio_estudante, 2), round(saldo_medio_nao_estudante, 2)

A explicação está no saldo: em média, um estudante deve 987,82 dólares no cartão; um não estudante, 771,77 — 216,05 dólares a menos.

In [ ]:
desvios = X_multiplo.std()
efeito_padronizado = coeficientes_multiplo * desvios

round(float(efeito_padronizado["saldo"]), 4), round(float(efeito_padronizado["renda"]), 4), round(float(efeito_padronizado["estudante_sim"]), 4)

Coeficientes em dólares, em dólares e num 0/1 não se comparam crus; multiplicar cada um pelo desvio-padrão do próprio preditor põe os três na mesma escala — log-chance por um desvio-padrão de variação. Nessa escala, saldo pesa 2,7748, contra 0,0405 de renda e -0,2948 de estudante: é saldo, não estudante, o preditor que mais empurra a inadimplência para cima, e é essa desvantagem que um estudante típico já carrega ao entrar na conta múltipla — o coeficiente isolado de `estudante` não separa os dois efeitos: sozinho, ele confunde o efeito de ser estudante com o efeito de, em média, dever mais.

In [ ]:
modelo_saldo_estudante = LogisticRegression(C=np.inf, max_iter=10000).fit(
    estudantes[["saldo"]], (estudantes["inadimplente"] == "sim").astype(int)
)
modelo_saldo_nao_estudante = LogisticRegression(C=np.inf, max_iter=10000).fit(
    nao_estudantes[["saldo"]], (nao_estudantes["inadimplente"] == "sim").astype(int)
)

grade_saldo = np.linspace(base["saldo"].min(), base["saldo"].max(), 300)
grade_saldo_df = pd.DataFrame({"saldo": grade_saldo})
prob_estudante = modelo_saldo_estudante.predict_proba(grade_saldo_df)[:, 1]
prob_nao_estudante = modelo_saldo_nao_estudante.predict_proba(grade_saldo_df)[:, 1]
estudante_sempre_abaixo = bool((prob_estudante < prob_nao_estudante).all())

faixa_comum_min = float(max(estudantes["saldo"].min(), nao_estudantes["saldo"].min()))
faixa_comum_max = float(min(estudantes["saldo"].max(), nao_estudantes["saldo"].max()))
grade_comum_df = pd.DataFrame({"saldo": np.linspace(faixa_comum_min, faixa_comum_max, 300)})
prob_estudante_comum = modelo_saldo_estudante.predict_proba(grade_comum_df)[:, 1]
prob_nao_estudante_comum = modelo_saldo_nao_estudante.predict_proba(grade_comum_df)[:, 1]
abaixo_na_faixa_comum = bool((prob_estudante_comum < prob_nao_estudante_comum).all())

base_decis = base.copy()
base_decis["decil_saldo"] = pd.qcut(base_decis["saldo"], 10)
taxas_por_decil = (
    base_decis.groupby(["decil_saldo", "estudante"], observed=True)["inadimplente"]
    .apply(lambda s: (s == "sim").mean())
    .unstack("estudante")
)
estudante_nunca_acima_no_decil = bool((taxas_por_decil["sim"] <= taxas_por_decil["não"]).all())

(
    estudante_sempre_abaixo,
    round(faixa_comum_min, 2),
    round(faixa_comum_max, 2),
    abaixo_na_faixa_comum,
    estudante_nunca_acima_no_decil,
)

Dois modelos logísticos ajustados separadamente — um só com os estudantes, outro só com os não estudantes, cada um usando `saldo` como único preditor — mostram a curva do estudante abaixo da do não estudante em toda a grade: `estudante_sempre_abaixo` sai `True`. O mesmo vale restrito à faixa de saldo que os dois grupos de fato ocupam, de 0,00 a 2.499,02 dólares (`abaixo_na_faixa_comum`). O dado cru confirma isso sem depender de modelo nenhum: dividindo `saldo` em dez decis, a taxa observada de inadimplência do estudante não passa a do não estudante em nenhum dos dez (`estudante_nunca_acima_no_decil`).

In [ ]:
# Figura: Esquerda: taxa de inadimplência contra saldo. As curvas cheias são estimadas por um modelo logístico ajustado separadamente em cada grupo (laranja: estudante; azul: não estudante); as tracejadas marcam a taxa bruta observada de cada grupo, sem olhar para saldo. Direita: distribuição de saldo por grupo — o estudante se concentra em saldos mais altos.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))

ax1.plot(grade_saldo, prob_nao_estudante, color="C0", linewidth=2.2, label="não")
ax1.plot(grade_saldo, prob_estudante, color="C1", linewidth=2.2, label="sim")
ax1.axhline(taxa_nao_estudante, color="C0", linestyle="--", linewidth=1.2, alpha=0.7)
ax1.axhline(taxa_estudante, color="C1", linestyle="--", linewidth=1.2, alpha=0.7)
ax1.annotate(
    "taxa bruta: não", xy=(grade_saldo[-1], taxa_nao_estudante),
    xytext=(-98, -10), textcoords="offset points", fontsize=8, color="C0",
)
ax1.annotate(
    "taxa bruta: sim", xy=(grade_saldo[-1], taxa_estudante),
    xytext=(-98, 6), textcoords="offset points", fontsize=8, color="C1",
)
ax1.set_xlabel("saldo (dólares)")
ax1.set_ylabel("taxa de inadimplência")
ax1.legend(title="estudante", loc="upper left")

for dados, posicao, cor in [(nao_estudantes["saldo"], 1, "C0"), (estudantes["saldo"], 2, "C1")]:
    propriedades = dict(color=cor, linewidth=1.6)
    ax2.boxplot(
        dados, positions=[posicao], widths=0.5,
        boxprops=propriedades, whiskerprops=propriedades,
        capprops=propriedades, medianprops=propriedades,
        flierprops=dict(markeredgecolor=cor, markersize=3, alpha=0.5),
    )
ax2.set_xticks([1, 2])
ax2.set_xticklabels(["não", "sim"])
ax2.set_xlabel("estudante")
ax2.set_ylabel("saldo (dólares)")

plt.tight_layout()
plt.show()

A curva da esquerda e os decis contam a mesma história: em qualquer saldo, o estudante não é o cliente mais arriscado — é a curva laranja que fica embaixo, do primeiro ao último ponto da grade. As linhas tracejadas, que ignoram saldo e mostram só a taxa bruta de cada grupo, invertem essa ordem porque o estudante se concentra em saldos mais altos — a diferença de 216,05 dólares na média medida acima — puxando a média geral dele para cima mesmo com cada saldo individual sendo mais seguro.

In [ ]:
prob_multiplo_todas = modelo_multiplo.predict_proba(X_multiplo)[:, 1]
mascara_estudante = X_multiplo["estudante_sim"] == 1

media_prob_estudante = float(prob_multiplo_todas[mascara_estudante].mean())
media_prob_nao_estudante = float(prob_multiplo_todas[~mascara_estudante].mean())

pd.DataFrame(
    {
        "prevista (média)": [media_prob_estudante, media_prob_nao_estudante],
        "bruta": [taxa_estudante, taxa_nao_estudante],
    },
    index=["estudante", "não estudante"],
).round(6)

A média da probabilidade que o modelo múltiplo prevê para cada estudante, sobre o saldo e a renda que ele de fato tem, é 0,043139; a mesma média entre não estudantes é 0,029195 — e a tabela põe as duas ao lado da taxa bruta de cada grupo, iguais até a sexta casa decimal.

## Logística Multinomial

> **📌 Nota**
>
> Esta seção corresponde à seção 4.3.5 de James et al. (2023).

A seção anterior ajustou uma curva logística para um alvo de duas classes, inadimplente ou não. `origem`, em `Auto`, tem três — americano, europeu e japonês —, e a curva em S da seção 9.2 não responde a uma pergunta de três respostas: ela devolve a probabilidade de uma classe contra a outra, e aqui sobra sempre uma terceira sem lugar nessa conta. É essa extensão, de duas classes para $K$, que a logística multinomial resolve.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

plt.style.use("estilo-figuras.mplstyle")

### A base, de novo

A saída é a mesma da seção 8.4: escolhe-se uma classe para servir de referência — a **base** —, e cada uma das outras ganha o seu próprio conjunto de coeficientes, lidos contra essa base. Com $K$ classes e uma base fixada, restam $K - 1$ conjuntos de coeficientes; para um preditor $X = (x_1, \ldots, x_p)$, a probabilidade de uma classe $k$ diferente da base, e da própria base, ficam

$$
\Pr(Y = k \mid X = x) = \frac{e^{\beta_{k0} + \beta_{k1} x_1 + \cdots + \beta_{kp} x_p}}{1 + \sum_{l \neq \text{base}} e^{\beta_{l0} + \beta_{l1} x_1 + \cdots + \beta_{lp} x_p}}, \qquad
\Pr(Y = \text{base} \mid X = x) = \frac{1}{1 + \sum_{l \neq \text{base}} e^{\beta_{l0} + \cdots + \beta_{lp} x_p}}.
$$

> **🔷 Conceito**
>
> Com $K$ categorias sem ordem natural, escolhe-se uma como base; as $K - 1$ restantes ganham coeficientes próprios, lidos como log-chance contra essa base — o mesmo gesto da seção 8.4, em que `drop_first` descartava uma categoria e as indicadoras liam-se contra ela.

O `scikit-learn`, porém, não ajusta esse formato. Ele usa uma parametrização diferente, chamada *softmax*, que trata as $K$ classes de forma simétrica — nenhuma vira base, e todas ganham o próprio conjunto de coeficientes:

$$
\Pr(Y = k \mid X = x) = \frac{e^{\beta_{k0} + \beta_{k1} x_1 + \cdots + \beta_{kp} x_p}}{\sum_{l=1}^{K} e^{\beta_{l0} + \beta_{l1} x_1 + \cdots + \beta_{lp} x_p}}, \qquad k = 1, \ldots, K.
$$

Essa forma tem $K$ conjuntos de coeficientes, não $K - 1$: é sobreparametrizada, porque somar a mesma constante a $\beta_{k0}, \beta_{k1}, \ldots, \beta_{kp}$ em toda classe $k$ não muda probabilidade nenhuma — a constante aparece em todo expoente do numerador e do denominador, e cancela na divisão. É essa sobra que a parametrização com base elimina, zerando de propósito a linha da classe escolhida; a softmax simplesmente deixa a sobra aí, espalhada pelas $K$ linhas. As duas formas dizem a mesma coisa, e o resto da seção mostra isso com o `Auto`, no lugar do exemplo do pronto-socorro que o livro-texto usa aqui.

### Três origens, cinco preditores

In [ ]:
auto = pd.read_csv("dados/Auto.csv")
n_total = len(auto)

auto["potencia"] = pd.to_numeric(auto["potencia"], errors="coerce")
n_com_interrogacao = int(auto["potencia"].isna().sum())

auto_limpo = auto.dropna(subset=["potencia"]).copy()
n_linhas = len(auto_limpo)

n_total, n_com_interrogacao, n_linhas

`dados/Auto.csv` tem 397 carros. `potencia` chega como texto, porque cinco linhas trazem `?` em vez de um número; `pd.to_numeric(auto["potencia"], errors="coerce")` converte a coluna para número e transforma esses cinco `?` em ausente, e `dropna()` descarta as cinco linhas ausentes — sobram 392.

In [ ]:
contagem_origem = auto_limpo["origem"].value_counts().sort_index()
contagem_origem

Das 392 linhas restantes, 245 são americanos (`origem` 1), 68 são europeus (`origem` 2) e 79 são japoneses (`origem` 3). 245 dos 392 carros — 62,5% do conjunto (245/392) — vêm dos Estados Unidos; as outras duas origens dividem o resto. Cinco preditores técnicos — `cilindrada`, `potencia`, `peso`, `aceleracao` e `ano` — tentam prever qual das três é a origem de cada carro.

### Uma linha de coeficientes por classe

In [ ]:
preditores = ["cilindrada", "potencia", "peso", "aceleracao", "ano"]
X = auto_limpo[preditores]
y = auto_limpo["origem"]

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    modelo = LogisticRegression(C=np.inf, max_iter=10000).fit(X, y)

n_iteracoes = int(modelo.n_iter_[0])
n_avisos = len(avisos)
acuracia = float(modelo.score(X, y))

n_iteracoes, n_avisos, round(acuracia, 4)

O ajuste, com o mesmo `C=np.inf` e `max_iter=10000` da seção 9.2, converge em 4.739 iterações; capturando todo aviso emitido durante o `fit`, `n_avisos` sai 0 — nenhum aviso de convergência, nem de nenhum outro tipo. Sobre as mesmas 392 linhas que o treinaram, o modelo acerta 0,7883 das previsões; não é desempenho em dado novo, só a fração de acerto sobre o dado que ele já viu — julgar um classificador em dado que ele não viu é assunto do capítulo 10.

In [ ]:
tabela_coeficientes = pd.DataFrame(
    modelo.coef_, index=pd.Index(modelo.classes_, name="origem"), columns=modelo.feature_names_in_
)
tabela_coeficientes.insert(0, "intercepto", modelo.intercept_)
tabela_coeficientes.round(4)

`coef_` vem com três linhas, não duas: `classes_` dá 1, 2 e 3, a mesma codificação de `origem`, e cada linha traz um coeficiente por coluna de `feature_names_in_`. Nenhuma das três é a base — é a parametrização softmax descrita acima, simétrica, que o `scikit-learn` ajusta por padrão sempre que o alvo tem mais de duas classes.

### A base muda, a previsão não

In [ ]:
classe_base = int(modelo.classes_[0])

coef_recentrado = modelo.coef_ - modelo.coef_[0]
intercepto_recentrado = modelo.intercept_ - modelo.intercept_[0]

tabela_recentrada = pd.DataFrame(
    coef_recentrado, index=pd.Index(modelo.classes_, name="origem"), columns=modelo.feature_names_in_
)
tabela_recentrada.insert(0, "intercepto", intercepto_recentrado)

tabela_lado_a_lado = pd.concat(
    {
        "parametrização softmax": tabela_coeficientes,
        f"contra a base (origem {classe_base})": tabela_recentrada,
    },
    axis=1,
)
tabela_lado_a_lado.round(4)

Subtrair a linha da origem 1 de toda linha de `coef_` — e o mesmo em `intercept_` — produz exatamente a forma com base descrita no começo desta seção: a linha da origem 1 zera por completo, e as outras duas passam a ler-se contra ela. É de novo o gesto do `drop_first` da seção 8.4, agora sobre três classes em vez de duas categorias, e sem reajustar modelo nenhum — a tabela acima só reorganiza os mesmos coeficientes que `ajuste-multinomial` já tinha produzido.

In [ ]:
logitos_recentrados = X.values @ coef_recentrado.T + intercepto_recentrado
exp_logitos = np.exp(logitos_recentrados)
probabilidades_manuais = exp_logitos / exp_logitos.sum(axis=1, keepdims=True)

probabilidades_sklearn = modelo.predict_proba(X)
diferenca_maxima = float(np.abs(probabilidades_manuais - probabilidades_sklearn).max())
probabilidades_batem = bool(np.allclose(probabilidades_manuais, probabilidades_sklearn))

previsao_manual = modelo.classes_[probabilidades_manuais.argmax(axis=1)]
previsao_sklearn = modelo.predict(X)
previsoes_batem = bool(np.array_equal(previsao_manual, previsao_sklearn))

f"{diferenca_maxima:.2e}", probabilidades_batem, previsoes_batem

Recalculando a probabilidade à mão a partir desses logitos recentrados — a exponencial de cada logito dividida pela soma da linha, a própria definição de softmax — e comparando com o `predict_proba` do ajuste original: a diferença máxima, sobre as 392 × 3 entradas, é 1,11e-15, e `probabilidades_batem` sai `True`. A previsão — a classe de maior probabilidade em cada linha — também bate: `previsoes_batem` sai `True`. Os coeficientes mudam de número; o que o modelo prevê para cada um dos 392 carros, não.

### Três classes não são uma escala

A seção 9.1 mostrou esse problema do lado da codificação: rotular derrame, overdose e convulsão como 1, 2 e 3 inventa uma ordem e uma distância que a lista de diagnósticos não tinha. A matriz de confusão do ajuste sobre `Auto` mostra o mesmo problema do lado do erro.

In [ ]:
y_previsto = modelo.predict(X)
matriz = confusion_matrix(y, y_previsto, labels=modelo.classes_)

matriz_rotulada = pd.DataFrame(
    matriz,
    index=pd.Index(modelo.classes_, name="origem verdadeira"),
    columns=pd.Index(modelo.classes_, name="origem prevista"),
)
matriz_rotulada

`confusion_matrix(y, y_previsto)` do `scikit-learn` devolve linha = origem verdadeira, coluna = origem prevista — por isso a tabela acima, e a figura a seguir, rotulam os dois eixos em vez de deixar por conta da memória.

In [ ]:
erros_1_2 = int(matriz[0, 1] + matriz[1, 0])
erros_2_3 = int(matriz[1, 2] + matriz[2, 1])
erros_1_3 = int(matriz[0, 2] + matriz[2, 0])

longe_supera_1_2 = bool(erros_1_3 > erros_1_2)

erros_1_2, erros_2_3, erros_1_3, longe_supera_1_2

Contagem crua, porém, não controla pelo tamanho de cada origem — 245, 68 e 79 carros, a contagem por origem impressa acima —, e um par que reúne mais carros tem mais chance de acumular erro só por isso.

In [ ]:
n_1_2 = int(contagem_origem[1] + contagem_origem[2])
n_2_3 = int(contagem_origem[2] + contagem_origem[3])
n_1_3 = int(contagem_origem[1] + contagem_origem[3])

taxa_1_2 = erros_1_2 / n_1_2
taxa_2_3 = erros_2_3 / n_2_3
taxa_1_3 = erros_1_3 / n_1_3

longe_supera_1_2_em_taxa = bool(taxa_1_3 > taxa_1_2)

round(taxa_1_2 * 100, 2), round(taxa_2_3 * 100, 2), round(taxa_1_3 * 100, 2), longe_supera_1_2_em_taxa

In [ ]:
# Figura: Matriz de confusão do ajuste sobre as 392 linhas de `Auto`, como bolhas: a área de cada bolha cresce com a raiz quadrada da contagem da célula (maior para célula mais frequente, sem ser proporcional a ela), e o número exato está escrito dentro dela. Azul marca acerto (a diagonal); verde marca erro entre origens vizinhas na numeração (1-2 ou 2-3); laranja marca erro entre os dois extremos da numeração (1-3). Eixo vertical: origem verdadeira; eixo horizontal: origem prevista.
rotulos = ["americano", "europeu", "japonês"]

grupos = {
    "acerto": ([], [], [], "C0"),
    "vizinha na numeração (1-2 ou 2-3)": ([], [], [], "C2"),
    "extremos da numeração (1-3)": ([], [], [], "C1"),
}
for i in range(3):
    for j in range(3):
        contagem = int(matriz[i, j])
        if i == j:
            chave = "acerto"
        elif abs(i - j) == 1:
            chave = "vizinha na numeração (1-2 ou 2-3)"
        else:
            chave = "extremos da numeração (1-3)"
        grupos[chave][0].append(j)
        grupos[chave][1].append(i)
        grupos[chave][2].append(contagem)

fig, ax = plt.subplots(figsize=(6.2, 5.4))
for rotulo_legenda, (xs, ys, contagens, cor) in grupos.items():
    tamanhos = [70 * np.sqrt(c) for c in contagens]
    ax.scatter(xs, ys, s=tamanhos, color=cor, zorder=3)
    for x_pos, y_pos, c in zip(xs, ys, contagens):
        ax.annotate(str(c), xy=(x_pos, y_pos), ha="center", va="center", fontsize=9, color="white", zorder=4)

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(rotulos)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(rotulos)
ax.set_xlabel("origem prevista")
ax.set_ylabel("origem verdadeira")
ax.set_xlim(-0.6, 2.6)
ax.set_ylim(2.6, -0.6)

# Marcadores da legenda em tamanho fixo: nesta figura, só a cor identifica a
# categoria — o tamanho de cada bolha do gráfico já está ocupado codificando
# a contagem daquela célula.
marcadores_legenda = [
    Line2D([], [], marker="o", linestyle="None", color=cor, markersize=9)
    for _, _, _, cor in grupos.values()
]
ax.legend(
    marcadores_legenda, list(grupos.keys()),
    title="leitura da célula", loc="upper center", bbox_to_anchor=(0.5, -0.18),
)
plt.tight_layout()
plt.show()

Se `origem` fosse mesmo uma escala — 1, depois 2, depois 3 —, o par de extremos (1-3) devia ser o mais raro de confundir, não um dos mais comuns. A contagem crua já aponta nessa direção: entre americano e japonês (1-3) o modelo erra 33 vezes (19 + 14), mais que entre americano e europeu (1-2), 17 vezes (7 + 10) — `longe_supera_1_2` sai `True`. Mas as três origens têm tamanhos bem diferentes (245, 68 e 79 carros), e um par que reúne mais carros acumula mais chance de erro só por isso, sem que a numeração tenha nada a ver. Dividindo o erro de cada par pelo total de carros que ele reúne — a taxa de confusão do par —, o quadro muda de escala mas não de conclusão: o par vizinho 1-2 erra em 5,43% dos casos, o par vizinho 2-3 em 22,45%, e o par de extremos, 1-3, em 10,19% — acima da taxa do par vizinho 1-2 (`longe_supera_1_2_em_taxa` sai `True`). O par mais distante na numeração de `origem` não é o mais fácil de separar: a numeração não é a régua que o erro segue.

## Modelos Generativos: LDA, QDA e Naive Bayes

> **📌 Nota**
>
> Esta seção corresponde à seção 4.4 de James et al. (2023).

A seção 9.2 modela $\Pr(Y \mid X)$ direto: ajusta a curva logística e lê o coeficiente sem nunca perguntar de onde vieram os valores de $X$ em cada classe. LDA, QDA e Naive Bayes fazem o caminho inverso — modelam como $X$ se distribui dentro de cada classe, e só depois usam o teorema de Bayes para virar essa distribuição em $\Pr(Y \mid X)$. A troca compensa em duas situações que deixam a logística mal-comportada: quando as classes estão bem separadas, os coeficientes da regressão logística ficam surpreendentemente instáveis; e quando há poucas observações mas a distribuição de $X$ dentro de cada classe é aproximadamente normal, assumir essa forma aproveita melhor o pouco dado do que estimar a fronteira direto, sem suposição nenhuma.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB

plt.style.use("estilo-figuras.mplstyle")

base = pd.read_csv("dados/Default.csv")
y = (base["inadimplente"] == "sim").astype(int)
X = base[["saldo", "renda"]].copy()
X["estudante_sim"] = (base["estudante"] == "sim").astype(int)

### Da distribuição de X à probabilidade de Y

Cada um dos três métodos parte da mesma pergunta: como $X$ se distribui dentro da classe $k$? Chame de $\pi_k$ a proporção de observações que vêm da classe $k$ antes de olhar para $X$ — a probabilidade *a priori* —, e de $f_k(x)$ a densidade de $X$ entre as observações da classe $k$: grande onde um $X$ daquela classe é comum, pequena onde é raro. O teorema de Bayes transforma isso na probabilidade que interessa, a de $Y$ dado $X$:

$$
\Pr(Y = k \mid X = x) = \frac{\pi_k f_k(x)}{\sum_{l=1}^{K} \pi_l f_l(x)}.
$$

$\pi_k$ é fácil de estimar — a fração das observações de treino que pertence à classe $k$. $f_k(x)$ é a parte difícil, e é exatamente aí que LDA, QDA e Naive Bayes discordam: cada um assume uma forma diferente para essa densidade, e é essa forma, e só ela, que decide o formato da fronteira entre as classes.

### LDA: a mesma covariância para todas as classes

LDA assume que, dentro de cada classe, $X$ segue uma normal multivariada em torno de uma média $\mu_k$ própria da classe, e que todas as classes **compartilham a mesma matriz de covariância** $\Sigma$. Plugando essa densidade no teorema de Bayes e tomando o log, a classe escolhida é a que maximiza

$$
\delta_k(x) = x^T \Sigma^{-1} \mu_k - \frac{1}{2} \mu_k^T \Sigma^{-1} \mu_k + \log \pi_k.
$$

> **🔷 Conceito**
>
> Com uma matriz de covariância só, o termo $x^T \Sigma^{-1} x$ que apareceria ao expandir a distância de $x$ até cada $\mu_k$ é o mesmo em toda classe — cancela ao comparar $\delta_k(x)$ com $\delta_l(x)$, e sobra apenas uma combinação linear de $x$. É daí, e só daí, que sai a fronteira **linear** que dá nome ao método.

### QDA: uma covariância por classe

QDA solta a suposição mais forte da LDA: cada classe passa a ter a sua própria matriz de covariância, $\Sigma_k$. A classe escolhida passa a maximizar

$$
\delta_k(x) = -\frac{1}{2}(x-\mu_k)^T \Sigma_k^{-1} (x-\mu_k) - \frac{1}{2}\log|\Sigma_k| + \log \pi_k.
$$

Sem uma covariância comum, o termo $x^T \Sigma_k^{-1} x$ não cancela mais entre classes — ele carrega o índice $k$ agora — e sobra uma função **quadrática** de $x$: a fronteira se curva.

A liberdade custa parâmetro: cada matriz de covariância com $p$ preditores tem $p(p+1)/2$ entradas livres a estimar; a LDA estima uma só, a QDA estima uma por classe.

In [ ]:
p = X.shape[1]
K = int(y.nunique())

parametros_lda = p * (p + 1) // 2
parametros_qda = K * p * (p + 1) // 2

p, K, parametros_lda, parametros_qda

Com os três preditores que o resto desta seção usa sobre o `Default` — saldo, renda e o indicador de estudante, $p=3$ — e as duas classes de inadimplência, $K=2$: a LDA estima 6 parâmetros de covariância; a QDA, 12, o dobro. É a mesma troca que a seção 7.3 descreveu entre um ajuste paramétrico rígido e um mais livre, só que dentro da própria família normal — menos parâmetros pede menos dado e trava a fronteira numa forma rígida (linear); mais parâmetros exige mais dado e libera uma fronteira mais flexível (quadrática).

### Naive Bayes: independência dentro da classe

Naive Bayes troca a suposição sobre a forma da densidade por uma suposição sobre a relação entre os preditores: dentro de cada classe, eles são **independentes**. A densidade conjunta se fatora num produto de densidades de um preditor só:

$$
f_k(x) = f_{k1}(x_1) \times f_{k2}(x_2) \times \cdots \times f_{kp}(x_p).
$$

Isso elimina de vez a parte mais cara de estimar $f_k$: a associação entre os preditores, que com muitos preditores e pouco dado é a mais difícil de estimar bem. A suposição quase nunca é exata — saldo e renda, por exemplo, não têm por que ser independentes dentro do grupo de quem fica inadimplente —, mas trocar uma associação difícil de estimar por presumir que ela não existe reduz a variância do ajuste mais do que costuma acrescentar de viés, e é por isso que o método funciona bem na prática mesmo quando a suposição que o batiza é falsa.

> **🔷 Conceito**
>
> | | O que assume sobre X dentro da classe | Fronteira |
> |---|---|---|
> | LDA | normal, uma covariância comum a todas as classes | linear |
> | QDA | normal, uma covariância própria por classe | quadrática |
> | Naive Bayes | preditores independentes dentro da classe | depende da densidade marginal escolhida para cada preditor |

### Uma suposição que acerta, e uma que erra

A suposição da LDA — uma covariância só — acerta quando as classes de fato compartilham a forma da nuvem de pontos, e erra quando não compartilham. Dado simulado, em que a covariância verdadeira de cada classe é escolhida por quem gera o dado, deixa comparar os dois casos lado a lado: duas classes com a mesma covariância num painel, com covariâncias diferentes no outro. Um sorteio de treino só não decide isso de forma confiável — a diferença entre LDA e QDA num único sorteio pequeno é ruído tanto quanto sinal —, então o que decide é repetir o sorteio muitas vezes contra o mesmo conjunto de teste.

In [ ]:
rng = np.random.default_rng(9)

media_azul = np.array([-1.5, -1.5])
media_laranja = np.array([1.5, 1.5])
cov_comum = np.array([[1.0, 0.7], [0.7, 1.0]])
cov_laranja_diferente = np.array([[1.0, -0.7], [-0.7, 1.0]])

n_treino_por_classe = 30
n_teste_por_classe = 20000
n_replicas = 200


def gera_duas_classes(cov_azul, cov_laranja, n, rng):
    azul = rng.multivariate_normal(media_azul, cov_azul, size=n)
    laranja = rng.multivariate_normal(media_laranja, cov_laranja, size=n)
    X_gerado = np.vstack([azul, laranja])
    y_gerado = np.concatenate([np.zeros(n), np.ones(n)])
    return X_gerado, y_gerado


# Um conjunto de teste grande e FIXO por painel, sorteado uma vez e nunca usado
# para ajustar nada: mede os n_replicas sorteios de treino a seguir contra o
# mesmo alvo, em vez de cada um contra o seu próprio.
X_teste_comum, y_teste_comum = gera_duas_classes(cov_comum, cov_comum, n_teste_por_classe, rng)
X_teste_diferente, y_teste_diferente = gera_duas_classes(
    cov_comum, cov_laranja_diferente, n_teste_por_classe, rng
)

erros_lda_comum = np.empty(n_replicas)
erros_qda_comum = np.empty(n_replicas)
erros_lda_diferente = np.empty(n_replicas)
erros_qda_diferente = np.empty(n_replicas)

for i in range(n_replicas):
    X_treino_comum, y_treino_comum = gera_duas_classes(cov_comum, cov_comum, n_treino_por_classe, rng)
    X_treino_diferente, y_treino_diferente = gera_duas_classes(
        cov_comum, cov_laranja_diferente, n_treino_por_classe, rng
    )

    lda_comum_i = LinearDiscriminantAnalysis().fit(X_treino_comum, y_treino_comum)
    qda_comum_i = QuadraticDiscriminantAnalysis().fit(X_treino_comum, y_treino_comum)
    lda_diferente_i = LinearDiscriminantAnalysis().fit(X_treino_diferente, y_treino_diferente)
    qda_diferente_i = QuadraticDiscriminantAnalysis().fit(X_treino_diferente, y_treino_diferente)

    erros_lda_comum[i] = 1 - lda_comum_i.score(X_teste_comum, y_teste_comum)
    erros_qda_comum[i] = 1 - qda_comum_i.score(X_teste_comum, y_teste_comum)
    erros_lda_diferente[i] = 1 - lda_diferente_i.score(X_teste_diferente, y_teste_diferente)
    erros_qda_diferente[i] = 1 - qda_diferente_i.score(X_teste_diferente, y_teste_diferente)

media_erro_lda_comum = float(erros_lda_comum.mean())
media_erro_qda_comum = float(erros_qda_comum.mean())
media_erro_lda_diferente = float(erros_lda_diferente.mean())
media_erro_qda_diferente = float(erros_qda_diferente.mean())

fracao_lda_vence_comum = float((erros_lda_comum < erros_qda_comum).mean())
fracao_qda_vence_diferente = float((erros_qda_diferente < erros_lda_diferente).mean())

(
    round(media_erro_lda_comum, 4), round(media_erro_qda_comum, 4), round(fracao_lda_vence_comum, 2),
    round(media_erro_lda_diferente, 4), round(media_erro_qda_diferente, 4), round(fracao_qda_vence_diferente, 2),
)

Repetindo o sorteio de treino 200 vezes — sempre com `n_treino_por_classe=30` — contra o mesmo conjunto de teste fixo de 20.000 pontos por classe em cada painel: no painel de covariância comum, a LDA erra em média 5,56% contra 5,72% da QDA, e a LDA erra menos em 73% dos 200 sorteios — uma vantagem sistemática, mas modesta. No painel de covariâncias diferentes, a LDA erra em média 2,66% contra 1,05% da QDA, e a QDA erra menos em 99% dos 200 sorteios — quase toda corrida. A suposição da LDA, quando vale, compensa pouco; quando não vale, o preço de mantê-la é grande.

Os mesmos dois cenários, mas com LDA e QDA ajustados agora sobre um treino bem maior — 1.000 pontos por classe, contra os 30 usados acima —, deixam a fronteira ajustada perto da verdade em vez de perto do ruído de uma amostra pequena; é essa fronteira, ao lado da fronteira de Bayes — a fronteira ótima, que sai direto das densidades normais verdadeiras que geraram o dado, sem ajustar modelo nenhum —, que a figura a seguir desenha.

In [ ]:
# Figura: Duas classes simuladas — azul e laranja, 1.000 pontos de treino cada, mesma separação entre médias nos dois painéis —, com a fronteira de Bayes (linha cinza pontilhada, calculada das densidades normais verdadeiras) e as fronteiras ajustadas de LDA (verde sólida) e QDA (roxa tracejada). Esquerda: as duas classes compartilham a mesma matriz de covariância (correlação 0,7) — a fronteira de Bayes é reta, e a do LDA quase a cobre. Direita: a laranja tem correlação -0,7 em vez de 0,7 — a fronteira de Bayes se curva, e é a do QDA que a acompanha de perto.
n_treino_figura_por_classe = 1000

X_treino_comum_fig, y_treino_comum_fig = gera_duas_classes(cov_comum, cov_comum, n_treino_figura_por_classe, rng)
X_treino_diferente_fig, y_treino_diferente_fig = gera_duas_classes(
    cov_comum, cov_laranja_diferente, n_treino_figura_por_classe, rng
)

lda_comum_fig = LinearDiscriminantAnalysis().fit(X_treino_comum_fig, y_treino_comum_fig)
qda_comum_fig = QuadraticDiscriminantAnalysis().fit(X_treino_comum_fig, y_treino_comum_fig)
lda_diferente_fig = LinearDiscriminantAnalysis().fit(X_treino_diferente_fig, y_treino_diferente_fig)
qda_diferente_fig = QuadraticDiscriminantAnalysis().fit(X_treino_diferente_fig, y_treino_diferente_fig)


def log_densidade_normal(pontos, media, cov):
    p = media.shape[0]
    inv_cov = np.linalg.inv(cov)
    log_det_cov = np.log(np.linalg.det(cov))
    diff = pontos - media
    forma_quadratica = np.einsum("ij,jk,ik->i", diff, inv_cov, diff)
    return -0.5 * p * np.log(2 * np.pi) - 0.5 * log_det_cov - 0.5 * forma_quadratica


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6), sharex=True, sharey=True)

grade_x = np.linspace(-4.5, 4.5, 300)
grade_y = np.linspace(-4.5, 4.5, 300)
malha_x, malha_y = np.meshgrid(grade_x, grade_y)
pontos_grade = np.column_stack([malha_x.ravel(), malha_y.ravel()])

paineis = [
    (ax1, X_treino_comum_fig, y_treino_comum_fig, lda_comum_fig, qda_comum_fig, cov_comum, "mesma covariância"),
    (
        ax2, X_treino_diferente_fig, y_treino_diferente_fig, lda_diferente_fig, qda_diferente_fig,
        cov_laranja_diferente, "covariâncias diferentes",
    ),
]
for ax, X_treino, y_treino, modelo_lda, modelo_qda, cov_laranja_verdadeira, titulo in paineis:
    previsao_lda_grade = modelo_lda.predict(pontos_grade).reshape(malha_x.shape)
    previsao_qda_grade = modelo_qda.predict(pontos_grade).reshape(malha_x.shape)
    diferenca_log_densidade = (
        log_densidade_normal(pontos_grade, media_azul, cov_comum)
        - log_densidade_normal(pontos_grade, media_laranja, cov_laranja_verdadeira)
    ).reshape(malha_x.shape)

    ax.scatter(X_treino[y_treino == 0, 0], X_treino[y_treino == 0, 1], color="C0", s=6, zorder=3)
    ax.scatter(X_treino[y_treino == 1, 0], X_treino[y_treino == 1, 1], color="C1", s=6, zorder=3)
    ax.contour(
        malha_x, malha_y, diferenca_log_densidade, levels=[0], colors="#6C757D", linewidths=1.8,
        linestyles="dotted",
    )
    ax.contour(malha_x, malha_y, previsao_lda_grade, levels=[0.5], colors="C2", linewidths=2.2)
    ax.contour(
        malha_x, malha_y, previsao_qda_grade, levels=[0.5], colors="C3", linewidths=2.2, linestyles="dashed"
    )
    ax.set_xlabel("x1")
    ax.set_title(titulo)

ax1.set_ylabel("x2")

legenda_fronteiras = [
    Line2D([], [], marker="o", linestyle="None", color="C0", markersize=7, label="classe azul"),
    Line2D([], [], marker="o", linestyle="None", color="C1", markersize=7, label="classe laranja"),
    Line2D([], [], color="#6C757D", linewidth=1.8, linestyle=":", label="fronteira de Bayes"),
    Line2D([], [], color="C2", linewidth=2.2, label="fronteira LDA"),
    Line2D([], [], color="C3", linewidth=2.2, linestyle="dashed", label="fronteira QDA"),
]
fig.legend(handles=legenda_fronteiras, loc="lower center", ncols=5, bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.tight_layout()
plt.show()

A fronteira de Bayes dá ao leitor uma régua para julgar as outras duas. No painel de covariância comum, ela é reta — as duas classes de fato compartilham $\Sigma$ —, e a fronteira do LDA praticamente a cobre; a do QDA se afasta dela, curvando para acompanhar uma diferença de covariância que não existe de verdade. No painel de covariâncias diferentes, a fronteira de Bayes se curva, e é a do QDA que a acompanha de perto; a do LDA, presa a ser reta, erra a curva inteira. É a mesma assimetria que a média sobre 200 sorteios mediu acima, agora visível: o preço de a LDA manter uma suposição errada é maior do que o preço de a QDA estimar uma covariância a mais que não precisava.

### Os três sobre o Default

Sobre o `Default`, com os mesmos três preditores usados acima — saldo, renda e o indicador de estudante —, `LinearDiscriminantAnalysis`, `QuadraticDiscriminantAnalysis` e `GaussianNB`, as três classes do `scikit-learn` correspondentes aos três métodos, ajustam LDA, QDA e Naive Bayes exatamente como definidos acima; a `GaussianNB` estima cada densidade marginal $f_{kj}$ como uma normal — a opção mais simples entre as que James et al. (2023) lista para um preditor quantitativo; as outras são não paramétricas, um histograma ou uma densidade por kernel em vez de uma forma fixa.

In [ ]:
lda = LinearDiscriminantAnalysis().fit(X, y)
qda = QuadraticDiscriminantAnalysis().fit(X, y)
nb = GaussianNB().fit(X, y)

erro_lda = float((lda.predict(X) != y).mean())
erro_qda = float((qda.predict(X) != y).mean())
erro_nb = float((nb.predict(X) != y).mean())
erro_trivial = float(y.mean())

round(erro_lda, 4), round(erro_qda, 4), round(erro_nb, 4), round(erro_trivial, 4)

In [ ]:
pd.DataFrame(
    {"erro sobre as próprias linhas do ajuste": [erro_lda, erro_qda, erro_nb, erro_trivial]},
    index=["LDA", "QDA", "Naive Bayes", 'sempre "não"'],
).round(4)

In [ ]:
qda_erra_menos_que_lda = bool(erro_qda < erro_lda)
lda_erra_menos_que_nb = bool(erro_lda < erro_nb)
diferenca_lda_qda = round(abs(erro_lda - erro_qda), 4)
todos_abaixo_do_trivial = bool(max(erro_lda, erro_qda, erro_nb) < erro_trivial)

qda_erra_menos_que_lda, lda_erra_menos_que_nb, diferenca_lda_qda, todos_abaixo_do_trivial

Entre os três, a QDA erra menos (2,70%) — `qda_erra_menos_que_lda` confirma —, mas por uma margem mínima sobre a LDA (2,76%): a diferença entre os dois é de apenas 0,0006, seis décimos de milésimo, perto demais para chamar de vitória. A LDA, por sua vez, erra menos que o Naive Bayes (2,93%), e `lda_erra_menos_que_nb` confirma. Prever "não" para todo mundo, sem olhar para saldo, renda ou estudante nenhum, erra 3,33%: os três classificadores, que de fato usam o dado, ficam abaixo disso — `todos_abaixo_do_trivial` confirma —, mas só um pouco. Essa proximidade não fecha a questão — o erro total esconde *como* se erra, e é esse *como* que a seção 9.5 mede.

## Avaliando um Classificador

> **📌 Nota**
>
> Esta seção corresponde à seção 4.4.2 de James et al. (2023).

A seção anterior ajustou três classificadores diferentes ao `Default` e mediu o erro de cada um sobre as próprias linhas do ajuste — os três abaixo do erro trivial, mas próximos demais entre si para decidir qualquer coisa só por essa proximidade. O erro total soma dois jeitos de errar bem diferentes: dizer "não" para quem de fato fica inadimplente, e dizer "sim" para quem não fica. Esta seção separa os dois, com o classificador logístico ajustado na seção 9.2 — `saldo`, `renda` e o indicador de `estudante` — como caso de trabalho. Como na seção 9.3, tudo o que segue é medido sobre as mesmas linhas que ajustaram o modelo, não em dado novo. O vocabulário que sai daqui vale para julgar qualquer classificador, não só este.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score

plt.style.use("estilo-figuras.mplstyle")

base = pd.read_csv("dados/Default.csv")
y = (base["inadimplente"] == "sim").astype(int)
X = base[["saldo", "renda"]].copy()
X["estudante_sim"] = (base["estudante"] == "sim").astype(int)

modelo = LogisticRegression(C=np.inf, max_iter=10000).fit(X, y)
proba = modelo.predict_proba(X)[:, 1]

### A acurácia engana quando a classe é rara

In [ ]:
n_total = len(y)
acerto_trivial = float((y == 0).mean())
erro_trivial = float(y.mean())

n_total, round(acerto_trivial, 4), round(erro_trivial, 4)

Um classificador que responde sempre "não", sem olhar `saldo`, `renda` nem `estudante`, acerta 96,67% das 10.000 linhas — porque 96,67% delas de fato não são inadimplentes; ele erra nos 3,33% restantes, e só neles. Esse número não mede o classificador: mede a proporção da classe rara. Qualquer classificador de verdade precisa ser julgado contra ele, não contra 100%, e ele por si só não distingue um classificador que aprendeu algo de um que só repete a classe mais comum.

### A matriz de confusão

In [ ]:
y_previsto_05 = modelo.predict(X)
matriz_05 = confusion_matrix(y, y_previsto_05, labels=[0, 1])

matriz_05_rotulada = pd.DataFrame(
    matriz_05,
    index=pd.Index(["não", "sim"], name="inadimplente verdadeiro"),
    columns=pd.Index(["não", "sim"], name="inadimplente previsto"),
)
matriz_05_rotulada

`modelo.predict(X)` usa o limiar padrão de 0,5: prevê "sim" quando a probabilidade ajustada passa de 0,5, "não" caso contrário. `confusion_matrix(y, y_previsto)` devolve linha = verdade, coluna = previsto — a mesma convenção da matriz da seção 9.3, e a que esta seção mantém do início ao fim. Chame as quatro células de VN (verdadeiro negativo), FP (falso positivo), FN (falso negativo) e VP (verdadeiro positivo), na ordem em que aparecem na tabela: linha "não", colunas "não" e "sim"; linha "sim", colunas "não" e "sim".

In [ ]:
vn_05, fp_05 = int(matriz_05[0, 0]), int(matriz_05[0, 1])
fn_05, vp_05 = int(matriz_05[1, 0]), int(matriz_05[1, 1])

erro_05 = (fp_05 + fn_05) / len(y)
precisao_05 = vp_05 / (vp_05 + fp_05)
revocacao_05 = vp_05 / (vp_05 + fn_05)

vn_05, fp_05, fn_05, vp_05, round(erro_05, 4), round(precisao_05, 4), round(revocacao_05, 4)

No limiar padrão: 9.627 acertos entre quem não fica inadimplente, contra 40 falsos alarmes; 105 acertos entre quem fica, contra 228 que passam despercebidos. O erro total — (FP + FN) dividido pelas 10.000 linhas — é 0,0268, abaixo dos 0,0333 do classificador trivial. Mas FP + FN mistura dois erros de custo bem diferente, e é aí que entram precisão e revocação:

$$
\text{precisão} = \frac{VP}{VP + FP}, \qquad \text{revocação} = \frac{VP}{VP + FN}.
$$

Precisão responde: entre quem o modelo acusou de inadimplente, quantos de fato eram — aqui, 105 entre 105 + 40 = 145, ou 0,7241. Revocação responde: entre quem de fato ficou inadimplente, quantos o modelo achou — 105 entre 105 + 228 = 333, ou 0,3153. O erro total de 2,68% parece ótimo até se perguntar de quem é o erro: no limiar padrão, o modelo acha 105 dos 333 inadimplentes (105 + 228 = 333) e deixa 228 passar sem sinalizar nada.

> **🔷 Conceito**
>
> Matriz de confusão, precisão e revocação são o vocabulário mínimo para julgar um classificador. A matriz separa os quatro jeitos de acertar e errar — VN, FP, FN, VP. Precisão (VP / (VP + FP)) mede a confiança de um alarme positivo: quando o modelo diz "sim", quão provável é que esteja certo. Revocação (VP / (VP + FN)) mede a cobertura sobre quem de fato é positivo: da classe que interessa achar, quanto o modelo de fato encontra. Nenhuma das duas depende do algoritmo que gerou a previsão — servem do mesmo jeito para regressão logística, LDA, QDA, Naive Bayes ou qualquer outro classificador.

### O limiar é escolha, não verdade

`predict` decide com um limiar fixo, 0,5, mas nada obriga a esse número: a mesma probabilidade ajustada, comparada contra um limiar mais baixo, muda quem entra em cada célula da matriz.

In [ ]:
def metricas(limiar):
    previsto = (proba >= limiar).astype(int)
    matriz = confusion_matrix(y, previsto, labels=[0, 1])
    vn, fp = int(matriz[0, 0]), int(matriz[0, 1])
    fn, vp = int(matriz[1, 0]), int(matriz[1, 1])
    erro = (fp + fn) / len(y)
    precisao = vp / (vp + fp) if (vp + fp) else float("nan")
    revocacao = vp / (vp + fn) if (vp + fn) else float("nan")
    return vn, fp, fn, vp, erro, precisao, revocacao


limiares_nomeados = [0.5, 0.2, 0.1]
resultados = [metricas(limiar) for limiar in limiares_nomeados]

tabela_limiares = pd.DataFrame(
    {
        "VN": [r[0] for r in resultados],
        "FP": [r[1] for r in resultados],
        "FN": [r[2] for r in resultados],
        "VP": [r[3] for r in resultados],
        "erro": [r[4] for r in resultados],
        "precisão": [r[5] for r in resultados],
        "revocação": [r[6] for r in resultados],
    },
    index=pd.Index(limiares_nomeados, name="limiar"),
)
tabela_limiares.round(4)

Baixar o limiar de 0,5 para 0,2 muda a matriz inteira: FP sobe de 40 para 277, FN cai de 228 para 130, VP sobe de 105 para 203. A revocação sobe de 0,3153 para 0,6096 — quase o dobro de inadimplentes encontrados —, e a precisão cai de 0,7241 para 0,4229, porque boa parte de quem passa a ser acusado agora não é, de fato, inadimplente. O erro total também sobe, de 0,0268 para 0,0407. Em 0,1, a revocação chega a 0,7447 e a precisão cai a 0,3069, com erro total de 0,0645. Não existe limiar que melhore revocação e precisão ao mesmo tempo nesta tabela: cada um dos três compra uma coisa pagando a outra.

Três limiares escolhidos à mão mostram a direção da troca, mas não dizem se ela vale para todo o intervalo entre eles. Varrendo o limiar de 0,0 a 0,5, em passos de 0,002:

In [ ]:
n_pos = int(y.sum())
n_neg = int((y == 0).sum())

grade_limiares = np.linspace(0.0, 0.5, 251)
erro_total = np.empty_like(grade_limiares)
taxa_perdidos = np.empty_like(grade_limiares)
taxa_falso_alarme = np.empty_like(grade_limiares)
fp_contagem = np.empty(len(grade_limiares), dtype=int)
fn_contagem = np.empty(len(grade_limiares), dtype=int)

for i, limiar in enumerate(grade_limiares):
    previsto = (proba >= limiar).astype(int)
    fp = int(((y == 0) & (previsto == 1)).sum())
    fn = int(((y == 1) & (previsto == 0)).sum())
    fp_contagem[i] = fp
    fn_contagem[i] = fn
    erro_total[i] = (fp + fn) / len(y)
    taxa_perdidos[i] = fn / n_pos
    taxa_falso_alarme[i] = fp / n_neg

perdidos_nunca_cai_com_limiar = bool((np.diff(taxa_perdidos) >= 0).all())
falso_alarme_nunca_sobe_com_limiar = bool((np.diff(taxa_falso_alarme) <= 0).all())

# Onde as duas TAXAS (denominadores 333 e 9.667) mais se aproximam.
indice_cruzamento_taxas = int(np.argmin(np.abs(taxa_perdidos - taxa_falso_alarme)))
limiar_cruzamento_taxas = float(grade_limiares[indice_cruzamento_taxas])

# Onde as duas CONTAGENS (mesma unidade: pessoas) mais se aproximam — um
# ponto diferente do cruzamento das taxas, de propósito.
indice_cruzamento_contagem = int(np.argmin(np.abs(fp_contagem - fn_contagem)))
limiar_cruzamento_contagem = float(grade_limiares[indice_cruzamento_contagem])

erro_minimo = float(erro_total.min())
limiares_no_minimo = [round(float(t), 3) for t in grade_limiares[erro_total == erro_total.min()]]

(
    perdidos_nunca_cai_com_limiar,
    falso_alarme_nunca_sobe_com_limiar,
    round(limiar_cruzamento_taxas, 3),
    int(fp_contagem[indice_cruzamento_taxas]),
    int(fn_contagem[indice_cruzamento_taxas]),
    round(limiar_cruzamento_contagem, 3),
    int(fp_contagem[indice_cruzamento_contagem]),
    int(fn_contagem[indice_cruzamento_contagem]),
    round(erro_minimo, 4),
    limiares_no_minimo,
    round(float(erro_total[-1]), 4),
)

Nos 251 limiares varridos, a fração de inadimplentes perdidos nunca cai quando o limiar sobe (`perdidos_nunca_cai_com_limiar` é `True`), e a fração de alarme falso entre adimplentes nunca sobe quando o limiar sobe (`falso_alarme_nunca_sobe_com_limiar` é `True`): as duas se movem em sentidos opostos o tempo todo, não só nos três pontos escolhidos acima. As duas *taxas* se cruzam perto do limiar 0,038 — mas são taxas com bases diferentes, 333 inadimplentes contra 9.667 adimplentes. Naquele mesmo limiar, em contagem de pessoas, o modelo produz 1.179 falsos alarmes contra apenas 40 inadimplentes perdidos: quase 30 alarmes falsos para cada inadimplente ainda não achado. Taxas iguais não são erros iguais — quem opera o cartão sente contagem, não taxa. As duas *contagens* só se equilibram bem mais adiante, perto do limiar 0,278, com 160 falsos alarmes contra 159 perdidos. E o erro total mínimo da varredura, 0,0261, ocorre em dois limiares ao mesmo tempo, 0,428 e 0,430 (`limiares_no_minimo`, empate exato — `erro_total` é uma contagem inteira dividida por 10.000, não arredondamento) — não exatamente em 0,5 (que dá 0,0268): perto de 0,5 o erro total já está quase plano.

In [ ]:
# Figura: Taxas de erro contra o limiar de decisão, de 0,0 a 0,5, para o classificador logístico do Default. Verde: erro total. Laranja: fração de inadimplentes (verdadeiros \"sim\") que o modelo deixa passar. Azul: fração de adimplentes (verdadeiros \"não\") que o modelo alarma por engano. As duas últimas se cruzam perto de 0,04 e se movem em direções opostas em toda a faixa.
fig, ax = plt.subplots()
ax.plot(grade_limiares, erro_total, color="C2", linewidth=2.2, label="erro total")
ax.plot(grade_limiares, taxa_perdidos, color="C1", linewidth=2.2, linestyle="--", label="inadimplentes perdidos")
ax.plot(grade_limiares, taxa_falso_alarme, color="C0", linewidth=2.2, linestyle=":", label="alarme falso entre adimplentes")
ax.scatter([limiar_cruzamento_taxas], [taxa_perdidos[indice_cruzamento_taxas]], color="#6C757D", zorder=5, s=28)
ax.annotate(
    f"limiar ≈ {limiar_cruzamento_taxas:.3f}",
    xy=(limiar_cruzamento_taxas, taxa_perdidos[indice_cruzamento_taxas]),
    xytext=(0.25, 0.25), textcoords="data", fontsize=8,
    arrowprops=dict(arrowstyle="-", color="#6C757D", linewidth=0.8),
)
ax.set_xlim(0, 0.5)
ax.set_ylim(0, 1)
ax.set_xlabel("limiar de decisão")
ax.set_ylabel("taxa")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

O que decide qual limiar usar não sai dessa figura sozinha: baixar o limiar sempre acha mais inadimplentes, sempre à custa de mais alarme falso, e nenhum ponto do gráfico é "o melhor" sem uma resposta para uma pergunta que a figura não faz — quanto custa, para quem opera o cartão de crédito, deixar passar um inadimplente comparado a incomodar um cliente em dia. Essa é uma pergunta sobre o problema, não sobre estatística, e por isso esta seção não escolhe um limiar — mostra o que cada escolha custa.

### A curva ROC

A varredura acima olha limiar por limiar. A curva ROC resume todos eles de uma vez: para cada limiar possível, marca a taxa de acerto entre inadimplentes (a revocação) contra a taxa de alarme falso entre adimplentes, sem nunca imprimir o limiar em si.

In [ ]:
fpr, tpr, limiares_roc = roc_curve(y, proba)
auc = roc_auc_score(y, proba)

n_proba_distintas = len(np.unique(proba))
limiar_roc_inicial = float(limiares_roc[0])
curva_nunca_abaixo_da_diagonal = bool((tpr >= fpr).all())

n_proba_distintas, len(fpr), limiar_roc_inicial, round(float(auc), 4), curva_nunca_abaixo_da_diagonal

`roc_curve` não devolve um ponto por probabilidade observada. As probabilidades ajustadas por `predict_proba` têm 10.000 valores distintos (`n_proba_distintas`, impresso acima) — o mesmo total das linhas do `Default`, mas são quantidades diferentes que aqui só coincidem por acaso: uma é o número de clientes, a outra é o número de valores únicos que o ajuste produziu para eles. Por padrão o método descarta os pontos colineares que não mudam a forma da curva (`drop_intermediate=True`) e antecede um limiar artificial, `inf` — o primeiro valor de `limiares_roc`, também impresso acima —, que não é probabilidade nenhuma: daí 440 pares, não 10.000. São os vértices que bastam para desenhar a curva inteira, não uma amostra de limiares. Em nenhum dos 440 a revocação fica abaixo da taxa de falso alarme (`curva_nunca_abaixo_da_diagonal` é `True`) — a curva não cruza para baixo da diagonal em ponto nenhum. `roc_auc_score` resume a curva inteira num número: a área sob ela, 0,9496 — perto do máximo de 1,0, e bem acima dos 0,5 que um classificador sem informação nenhuma sobre `saldo`, `renda` ou `estudante` produziria.

In [ ]:
# Figura: Curva ROC do classificador logístico sobre o Default (azul): taxa de falso alarme (FPR) contra revocação (TPR), para os 440 limiares que definem a curva. A diagonal pontilhada cinza marca o classificador sem informação nenhuma sobre saldo, renda ou estudante — FPR e revocação crescendo juntos, um a um. O ponto laranja marca o limiar padrão, 0,5, já visto na matriz de confusão. A área sob a curva azul, 0,9496, está na legenda.
fpr_05 = fp_05 / n_neg

fig, ax = plt.subplots()
ax.plot(fpr, tpr, color="C0", linewidth=2.2, label=f"curva ROC (regressão logística) — AUC = {auc:.4f}")
ax.plot([0, 1], [0, 1], color="#7A8894", linestyle=":", linewidth=1.4, label="palpite sem informação")
ax.scatter([fpr_05], [revocacao_05], color="C1", zorder=5, s=40)
ax.annotate(
    f"limiar 0,5 (FPR {fpr_05:.3f}, revocação {revocacao_05:.3f})",
    xy=(fpr_05, revocacao_05), xytext=(0.15, 0.6), textcoords="data", fontsize=8,
    arrowprops=dict(arrowstyle="-", color="C1", linewidth=0.8),
)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("taxa de falso alarme (FPR)")
ax.set_ylabel("revocação (TPR)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

O ponto marcado é o limiar 0,5 já visto na matriz de confusão: FPR de 0,004 (os 40 falsos alarmes sobre 9.627 + 40 = 9.667 adimplentes) e revocação de 0,315 (os 105 acertos sobre 105 + 228 = 333 inadimplentes) — o mesmo par de números da seção anterior, agora lido como um ponto sobre a curva inteira. A diagonal pontilhada marca o que um classificador sem informação produziria: ganhar revocação só à custa de FPR, na mesma proporção, um a um. A curva do modelo logístico nunca cruza para baixo dela, como o `curva_nunca_abaixo_da_diagonal` do chunk acima confirma, e é a distância entre as duas — resumida na área de 0,9496 — que separa um classificador que aprendeu algo de um que está adivinhando.

A matriz de confusão, a precisão, a revocação e a curva ROC não dependem de qual classificador as produziu: servem para LDA, QDA, Naive Bayes e a regressão logística do mesmo jeito. É esse o vocabulário que permite comparar classificadores, e a seção 9.6 fecha o capítulo perguntando quem ganha quando.

## Comparando os Métodos

> **📌 Nota**
>
> Esta seção corresponde à seção 4.5 de James et al. (2023).

A seção 9.5 fechou perguntando quem ganha quando. Cinco métodos entraram neste capítulo: a regressão logística da seção 9.2, LDA, QDA e Naive Bayes da 9.4, e o *k*-NN, que as seções 7.3, 7.7 e 8.7 já aplicaram tanto à regressão quanto à classificação. Nesta seção o *k*-NN entra duas vezes — com *k*=1 e com *k*=15, tratados como dois classificadores distintos, porque é o que são —, o que dá **seis classificadores** medidos ao todo. A resposta honesta é "depende", e o resto desta seção mostra de quê: cenários simulados, com a fronteira verdadeira entre as duas classes conhecida de antemão — porque quem simula escolhe a distribuição de cada classe —, uns em que essa fronteira é linear e um em que não é.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("estilo-figuras.mplstyle")

base = pd.read_csv("dados/Default.csv")
y_default = (base["inadimplente"] == "sim").astype(int)
X_default = base[["saldo", "renda"]].copy()
X_default["estudante_sim"] = (base["estudante"] == "sim").astype(int)

k_pequeno, k_grande = 1, 15

### Seis classificadores, três cenários

O *k* de cada *k*-NN é fixo e declarado agora — `k_pequeno = 1` e `k_grande = 15` —, não ajustado ao próprio dado: cada valor de *k* é, na prática, um classificador diferente, e entra na tabela como tal, em vez de um só "*k*-NN" escolhido às pressas.

Como a seção 9.4 mostrou, um sorteio de treino só não compara métodos de forma confiável — a comparação que decide é a média sobre muitos sorteios, contra um conjunto de teste grande e fixo. O teste ser grande e fixo dentro de cada cenário existe para que a comparação não dependa de qual sorteio de teste calhou; a variação que importa é a do sorteio de **treino**, e é isso — não o teste — que a fração de vitórias, adiante, mede. `n_teste_por_classe = 5000` e `n_replicas = 200`, os mesmos 200 sorteios da seção 9.4.

In [ ]:
rng = np.random.default_rng(9)

media_a = np.array([-1.0, 0.0])
media_b = np.array([1.0, 0.0])

cov_independente = np.eye(2)
cov_correlacionada = np.array([[1.0, 0.8], [0.8, 1.0]])
cov_a_curva = np.array([[1.0, 0.7], [0.7, 1.0]])
cov_b_curva = np.array([[1.0, -0.7], [-0.7, 1.0]])

n_teste_por_classe = 5000
n_replicas = 200


def gera_duas_classes(cov_a, cov_b, n, rng):
    a = rng.multivariate_normal(media_a, cov_a, size=n)
    b = rng.multivariate_normal(media_b, cov_b, size=n)
    X = np.vstack([a, b])
    y = np.concatenate([np.zeros(n), np.ones(n)])
    return X, y


def constroi_metodos():
    return {
        "Logística": LogisticRegression(C=np.inf, max_iter=10000),
        "LDA": LinearDiscriminantAnalysis(),
        "QDA": QuadraticDiscriminantAnalysis(),
        "Naive Bayes": GaussianNB(),
        f"k-NN (k={k_pequeno})": KNeighborsClassifier(n_neighbors=k_pequeno),
        f"k-NN (k={k_grande})": KNeighborsClassifier(n_neighbors=k_grande),
    }


nomes_metodos = list(constroi_metodos().keys())

# fronteira curva usa mais dado de treino: k-NN só compensa a variância que
# sua flexibilidade custa quando há dado suficiente (seções 7.3 e 8.7).
cenarios = {
    "linear, preditores independentes": (cov_independente, cov_independente, 20),
    "linear, preditores correlacionados": (cov_correlacionada, cov_correlacionada, 20),
    "fronteira curva": (cov_a_curva, cov_b_curva, 200),
}

linhas = []
for nome_cenario, (cov_a, cov_b, n_treino_por_classe) in cenarios.items():
    X_teste, y_teste = gera_duas_classes(cov_a, cov_b, n_teste_por_classe, rng)
    for replica in range(n_replicas):
        X_treino, y_treino = gera_duas_classes(cov_a, cov_b, n_treino_por_classe, rng)
        for nome_metodo, modelo in constroi_metodos().items():
            modelo.fit(X_treino, y_treino)
            erro = 1 - modelo.score(X_teste, y_teste)
            linhas.append((nome_cenario, nome_metodo, replica, erro))

resultados = pd.DataFrame(linhas, columns=["cenario", "metodo", "replica", "erro"])
tabela_erro_medio = (
    resultados.groupby(["cenario", "metodo"])["erro"].mean().unstack("metodo")
    .reindex(index=list(cenarios.keys()), columns=nomes_metodos)
)
tabela_erro_medio.round(4)

Três cenários, cada um com dois preditores e duas classes com médias `media_a = (-1, 0)` e `media_b = (1, 0)` — só a primeira coordenada separa as classes, a segunda tem a mesma distribuição nas duas. O que muda de um cenário a outro é só a covariância dentro de cada classe: idêntica e sem correlação no primeiro, idêntica e com correlação 0,8 no segundo, com sinais opostos (0,7 e -0,7, a mesma diferença de covariância que a seção 9.4 usa) no terceiro — e é só essa terceira mudança que torna a fronteira de Bayes curva em vez de reta. Duzentos sorteios de treino por cenário, cada um refazendo o ajuste dos seis classificadores contra o mesmo teste fixo daquele cenário.

In [ ]:
# Figura: Distribuição do erro de teste em 200 sorteios de treino, para os seis classificadores, nos três cenários simulados (painéis, cada um com sua própria escala de erro). Em laranja, o classificador com menor erro médio naquele cenário; em azul, os demais. A cor marca só essa distinção; a identidade de cada classificador está no rótulo do eixo.
ordem_metodos = ["LDA", "Logística", "QDA", "Naive Bayes", f"k-NN (k={k_pequeno})", f"k-NN (k={k_grande})"]
vencedor_por_cenario = {nome_cenario: tabela_erro_medio.loc[nome_cenario].idxmin() for nome_cenario in cenarios}
titulo_por_cenario = {
    "linear, preditores independentes": "linear, independentes",
    "linear, preditores correlacionados": "linear, correlacionados",
    "fronteira curva": "fronteira curva",
}

fig, eixos = plt.subplots(1, 3, figsize=(13, 4.8), sharey=True)
for ax, nome_cenario in zip(eixos, cenarios.keys()):
    dados_cenario = [
        resultados.loc[(resultados["cenario"] == nome_cenario) & (resultados["metodo"] == m), "erro"]
        for m in ordem_metodos
    ]
    caixas = ax.boxplot(
        dados_cenario, orientation="horizontal", tick_labels=ordem_metodos, widths=0.6,
        medianprops=dict(linewidth=1.8), boxprops=dict(linewidth=1.8),
        whiskerprops=dict(linewidth=1.4), capprops=dict(linewidth=1.4),
        flierprops=dict(markersize=3),
    )
    for i, nome_metodo in enumerate(ordem_metodos):
        cor = "#D9480F" if nome_metodo == vencedor_por_cenario[nome_cenario] else "#4195D1"
        caixas["boxes"][i].set_color(cor)
        caixas["medians"][i].set_color(cor)
        caixas["whiskers"][2 * i].set_color(cor)
        caixas["whiskers"][2 * i + 1].set_color(cor)
        caixas["caps"][2 * i].set_color(cor)
        caixas["caps"][2 * i + 1].set_color(cor)
        caixas["fliers"][i].set_markeredgecolor(cor)
    ax.set_title(titulo_por_cenario[nome_cenario])
    ax.set_xlabel("erro de teste")

legenda_cores = [
    Line2D([], [], color="#4195D1", linewidth=1.8, label="demais classificadores"),
    Line2D([], [], color="#D9480F", linewidth=1.8, label="menor erro médio no cenário"),
]
fig.legend(handles=legenda_cores, loc="lower center", ncols=2, bbox_to_anchor=(0.5, -0.06), frameon=False)
plt.tight_layout()
plt.show()

A tabela e a figura contam a mesma história de dois jeitos: a tabela dá a média de cada caixa, a figura dá a caixa inteira — e é a caixa, não só a média, que diz se uma vantagem é grande o bastante para não desaparecer no próximo sorteio. No primeiro painel, as caixas de LDA, logística, QDA e Naive Bayes se sobrepõem bastante. No segundo, essa sobreposição vale só para LDA, QDA e logística — a caixa do Naive Bayes se descola por completo, a mesma lição que o cenário seguinte mede em número. No terceiro painel, a caixa de QDA não toca a de nenhuma das outras cinco.

### Fronteira linear, preditores independentes

In [ ]:
piv_indep = resultados[resultados["cenario"] == "linear, preditores independentes"].pivot(
    index="replica", columns="metodo", values="erro"
)

media_lda_indep = float(tabela_erro_medio.loc["linear, preditores independentes", "LDA"])
media_log_indep = float(tabela_erro_medio.loc["linear, preditores independentes", "Logística"])
media_nb_indep = float(tabela_erro_medio.loc["linear, preditores independentes", "Naive Bayes"])
media_qda_indep = float(tabela_erro_medio.loc["linear, preditores independentes", "QDA"])
media_knn1_indep = float(tabela_erro_medio.loc["linear, preditores independentes", f"k-NN (k={k_pequeno})"])
media_knnk_indep = float(tabela_erro_medio.loc["linear, preditores independentes", f"k-NN (k={k_grande})"])

margem_lda_qda_indep = media_qda_indep - media_lda_indep
fracao_lda_vence_qda_indep = float((piv_indep["LDA"] < piv_indep["QDA"]).mean())
amplitude_grupo_lider_indep = (
    max(media_lda_indep, media_log_indep, media_nb_indep, media_qda_indep)
    - min(media_lda_indep, media_log_indep, media_nb_indep, media_qda_indep)
)

(
    round(media_lda_indep, 4), round(media_log_indep, 4), round(media_nb_indep, 4), round(media_qda_indep, 4),
    round(media_knnk_indep, 4), round(media_knn1_indep, 4),
    round(margem_lda_qda_indep, 4), round(fracao_lda_vence_qda_indep, 2), round(amplitude_grupo_lider_indep, 4),
)

Vinte pontos de treino por classe, preditores sem correlação dentro de cada classe, a mesma covariância nas duas — exatamente a suposição da LDA. LDA, logística, Naive Bayes e QDA ficam a 0,0054 uma da outra: 17,43%, 17,59%, 17,58% e 17,97% de erro médio, nessa ordem — um grupo apertado, sem ninguém disparando na frente. *k*-NN com *k*=15 fica pouco acima desse grupo, 18,04%; *k*-NN com *k*=1 é quem de fato destoa, 23,74% — mais de cinco pontos percentuais acima até do pior do grupo de quatro (17,97% da QDA), o preço de uma vizinhança de um só ponto quando o treino tem só vinte por classe.

Dentro desse grupo apertado, uma comparação se sustenta: LDA erra menos que QDA em 81% dos 200 sorteios — uma vantagem sistemática, mas modesta, sobre uma margem estreita entre as médias (0,0054). É essa margem estreita que decide qual caixa a figura anterior pinta de laranja neste painel: o realce marca a menor média entre as quatro, não uma vantagem grande o bastante para chamar de vencedora do cenário. Entre LDA e Naive Bayes, porém, a distância é menor ainda, e esta seção não afirma qual dos dois vence: os dois pertencem ao mesmo grupo apertado, e o cenário seguinte mostra por quê Naive Bayes consegue chegar tão perto aqui.

### Fronteira ainda linear, preditores correlacionados

In [ ]:
piv_corr = resultados[resultados["cenario"] == "linear, preditores correlacionados"].pivot(
    index="replica", columns="metodo", values="erro"
)

media_lda_corr = float(tabela_erro_medio.loc["linear, preditores correlacionados", "LDA"])
media_qda_corr = float(tabela_erro_medio.loc["linear, preditores correlacionados", "QDA"])
media_log_corr = float(tabela_erro_medio.loc["linear, preditores correlacionados", "Logística"])
media_nb_corr = float(tabela_erro_medio.loc["linear, preditores correlacionados", "Naive Bayes"])

fracao_lda_vence_nb_corr = float((piv_corr["LDA"] < piv_corr["Naive Bayes"]).mean())

(
    round(media_lda_corr, 4), round(media_qda_corr, 4), round(media_log_corr, 4), round(media_nb_corr, 4),
    round(fracao_lda_vence_nb_corr, 2),
)

A fronteira continua linear — mesmas médias, a única mudança é a correlação dentro de cada classe: 0,8 em vez de zero. LDA, QDA e logística caem para 5,15%, 5,49% e 6,06% — a correlação carrega informação sobre a classe que esses três exploram, porque a direção que separa as classes, $\Sigma^{-1}(\mu_b - \mu_a)$, deixa de apontar só para a coordenada que muda de média e passa a misturar a outra também, mesmo ela tendo a mesma distribuição nas duas classes. Naive Bayes não tem como enxergar isso: ele trata as duas coordenadas como independentes por definição.

No cenário anterior, Naive Bayes errava 17,58% — dentro do grupo apertado descrito ali. Aqui, com a mesma suposição de independência violada, ele erra 15,95%: continua na mesma faixa alta de antes, enquanto LDA, QDA e logística caem para a faixa de 5% a 6%. A suposição de independência quase não separa Naive Bayes do resto quando ela vale; quando não vale, é o resto que se afasta dela. LDA erra menos que Naive Bayes em 100% dos 200 sorteios.

### Fronteira curva

In [ ]:
piv_curva = resultados[resultados["cenario"] == "fronteira curva"].pivot(
    index="replica", columns="metodo", values="erro"
)

media_qda_curva = float(tabela_erro_medio.loc["fronteira curva", "QDA"])
media_lda_curva = float(tabela_erro_medio.loc["fronteira curva", "LDA"])
media_log_curva = float(tabela_erro_medio.loc["fronteira curva", "Logística"])
media_nb_curva = float(tabela_erro_medio.loc["fronteira curva", "Naive Bayes"])
media_knnk_curva = float(tabela_erro_medio.loc["fronteira curva", f"k-NN (k={k_grande})"])
media_knn1_curva = float(tabela_erro_medio.loc["fronteira curva", f"k-NN (k={k_pequeno})"])

metodos_lineares_curva = ["LDA", "Logística", "Naive Bayes"]
melhor_linear_curva = tabela_erro_medio.loc["fronteira curva", metodos_lineares_curva].idxmin()
media_melhor_linear_curva = float(tabela_erro_medio.loc["fronteira curva", melhor_linear_curva])

fracao_qda_vence_knnk_curva = float((piv_curva["QDA"] < piv_curva[f"k-NN (k={k_grande})"]).mean())
fracao_knnk_vence_melhor_linear_curva = float(
    (piv_curva[f"k-NN (k={k_grande})"] < piv_curva[melhor_linear_curva]).mean()
)
margem_qda_proximo_curva = media_knnk_curva - media_qda_curva
margem_knnk_grupo_linear_curva = media_melhor_linear_curva - media_knnk_curva

(
    round(media_qda_curva, 4), round(media_knnk_curva, 4), round(media_log_curva, 4), round(media_lda_curva, 4),
    round(media_nb_curva, 4), round(media_knn1_curva, 4),
    round(fracao_qda_vence_knnk_curva, 2), round(fracao_knnk_vence_melhor_linear_curva, 2),
    round(margem_qda_proximo_curva, 4), round(margem_knnk_grupo_linear_curva, 4),
)

Este cenário usa 200 pontos de treino por classe, não os 20 dos dois anteriores: com covariâncias diferentes entre as classes, a fronteira de Bayes se curva, e tanto QDA quanto *k*-NN precisam de mais dado para essa curvatura compensar a variância que ela custa — as seções 7.3 e 8.7 já mostraram esse preço para regressão, e ele vale igual para classificação. QDA erra 13,31% em média, o menor de todos; *k*-NN com *k*=15 vem em segundo, 14,60%; logística, LDA e Naive Bayes empatam por perto, praticamente os três em 15,65%; *k*-NN com *k*=1 volta a ser o pior, 17,23%.

As duas comparações que separam os grupos são sólidas: QDA erra menos que *k*-NN (*k*=15) — seu concorrente mais próximo — em 99% dos 200 sorteios, uma margem de 0,0128 entre as médias; e *k*-NN (*k*=15) erra menos que o melhor dos três métodos lineares — praticamente empatados entre si, todos perto de 15,65% — em 94% dos 200 sorteios também, margem de 0,0105. Onde a fronteira verdadeira se curva, os dois métodos capazes de acompanhar uma curva — um assumindo a forma quadrática certa, o outro sem assumir forma nenhuma, desde que haja dado — ficam à frente dos três que só sabem traçar uma reta.

### Os seis classificadores sobre o Default

O `Default` fecha o capítulo com dado real, nas mesmas linhas que ajustaram os modelos da seção 9.4 — `saldo`, `renda` e o indicador de `estudante`.

In [ ]:
X_default[["saldo", "renda"]].agg(["mean", "std"]).round(1)

`saldo` tem desvio padrão de 483,7 dólares; `renda`, 13.336,6 — 27,6 vezes maior, mesmo as duas estando em dólares. *k*-NN decide "vizinho mais próximo" por distância, e nessa escala a distância seria dominada por `renda`, do mesmo jeito que a seção 7.3 já viu com escolaridade e senioridade em `Income2`: os dois *k*-NN desta tabela entram num `Pipeline` com `StandardScaler`, para que `saldo` tenha voz na conta.

In [ ]:
metodos_default = {
    "Logística": LogisticRegression(C=np.inf, max_iter=10000),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(),
    "Naive Bayes": GaussianNB(),
    f"k-NN (k={k_pequeno})": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k_pequeno)),
    f"k-NN (k={k_grande})": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k_grande)),
}

erros_default = {}
for nome_metodo, modelo in metodos_default.items():
    modelo.fit(X_default, y_default)
    erros_default[nome_metodo] = float((modelo.predict(X_default) != y_default).mean())

tabela_default = pd.DataFrame(
    {"erro sobre as próprias linhas do ajuste": erros_default}
).reindex(nomes_metodos)
tabela_default.round(4)

Como nas seções 9.4 e 9.5, o erro aqui é sobre as mesmas linhas que ajustaram cada modelo, não sobre dado novo. Logística, QDA, LDA e Naive Bayes ficam entre 2,68% e 2,93%, cotovelo a cotovelo. *k*-NN com *k*=1 erra 0%: cada cliente é seu próprio vizinho mais próximo nas 10.000 linhas que o ajustaram, então a previsão sempre bate com o rótulo que já tinha — o mesmo *k*=1 que decorou o treino nas seções 7.3, 7.7 e 8.7, e que não diz nada sobre um cliente novo.

In [ ]:
knnk_sem_escala = KNeighborsClassifier(n_neighbors=k_grande).fit(X_default, y_default)
erro_knnk_sem_escala = float((knnk_sem_escala.predict(X_default) != y_default).mean())
erro_knnk_com_escala = erros_default[f"k-NN (k={k_grande})"]

round(erro_knnk_sem_escala, 4), round(erro_knnk_com_escala, 4)

*k*-NN com *k*=15 mostra o preço da escala: sem `StandardScaler`, 3,18% de erro; com ele, 2,59% — quase seis décimos de ponto percentual só por colocar `saldo` e `renda` na mesma régua antes de medir distância.

Esse 2,59% é, na verdade, o menor erro não trivial de toda a tabela — abaixo dos quatro que ficam entre 2,68% e 2,93%. O mecanismo é o mesmo do *k*=1, só que menos óbvio: com as mesmas 10.000 linhas usadas para ajustar e para prever, cada cliente está entre os seus próprios 15 vizinhos mais próximos. Nos três cenários simulados, com um teste separado do treino, o quadro se inverte: *k*-NN (*k*=15) perde para LDA e para QDA em dois dos três — 18,04% contra 17,43% no primeiro, 10,98% contra 5,15% no segundo.

> **🔷 Conceito**
>
> | Cenário | O que se destacou | Do que depende |
> |---|---|---|
> | linear, preditores independentes | LDA, logística, QDA e Naive Bayes ficam num grupo apertado, sem vencedor destacado | nenhuma das quatro suposições é violada aqui |
> | linear, preditores correlacionados | LDA, QDA e logística caem juntos; Naive Bayes fica para trás | só o Naive Bayes assume preditores independentes dentro da classe |
> | fronteira curva, dado suficiente | QDA à frente; *k*-NN com *k* maior em segundo | são os dois capazes de acompanhar uma curva |

Nenhum dos seis classificadores vence em todo cenário, e qual deles ajustar a um problema novo depende de uma fronteira que, fora de uma simulação, ninguém conhece de antemão. Decidir entre eles sem essa certeza — e escolher o próprio *k* do *k*-NN, em vez de fixá-lo como esta seção fixou — é o que a validação cruzada, no capítulo 10, ensina a fazer.

## Leituras adicionais

*A escrever.*

## Referências

- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.